# GEO Photometry (satellite tracking)
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Measure brightness of GEO object captured with satellite tracking method.<br>
<br>
**History**<br>
coding 2026-07-06 : 1st coding<br>
update 2026-07-07 : support multi processing<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Input files settings**

In [ ]:
from pathlib import Path

# Input format: "SER" or "FITS"
input_format = "SER"

# For SER input, specify one SER file. For FITS input, specify a directory.
PATH_input   = Path("/Volumes/SSD-PEU4A/Images/2026-04-05/EKRAN 12/19_45_10.ser")
PATH_output  = Path("/Volumes/SSD-PEU4A/VScode/satphotometry_photometry/output")
PATH_magzero = Path("/Users/kiyoaki/VScode/satphotometry_photometry/output/magzero/magzero_2026-04-05T104510_GAIN200.json")

**Target and observation information**

In [ ]:
# Artificial-object information
object_name = "EKRAN 12"
intl_des    = "1984-028A"
norad_id    = "14821"

# Observer and observatory information
observer_name = "Kyushu University Hanada Lab (SSDL)"
observatory   = "KUBO Kyushu University Bull's-eye Telescope"

**SER files settings**

In [ ]:
# SER frame range (1-based, inclusive). Use None for the first/last frame.
ser_start_frame = None
ser_end_frame = None

# Camera settings used when converting SER to FITS.
# Values found in a SharpCap CameraSettings.txt file override these values.
ser_exposure_seconds = 1.0 / 8.0
ser_gain = None
ser_egain = None
ser_egainsav = None
ser_offset = None
ser_x_binning = 1
ser_y_binning = 1
ser_camera_name = None
ser_camera_id = None
ser_ccd_temperature = None
ser_set_temperature = None

**Calibration settings**

In [ ]:
# Calibration files. Set to None when not used.
PATH_dark = None
PATH_flat = None
PATH_flat_dark = None


**Plate solve settings**

In [ ]:
# Astrometry.net settings
run_astrometry = True
astrometry_downsample = 2
astrometry_search_radius_deg = 2.0
astrometry_scale_low_arcsec_per_pixel = None
astrometry_scale_high_arcsec_per_pixel = None
astrometry_extra_options = {
    "--objs": 50,
    "--crpix-center": True,
    "--no-remove-lines": True,
    "--uniformize": 0,
    "--overwrite": True,
    "--no-plots": True
}
astrometry_parallel = True
astrometry_max_workers = 4

**Star subtraction settings**

In [ ]:
# Star subtraction settings
subtract_stars = True
star_reference_interval = 16
star_shift_order = 3
star_shift_mode = "constant"
star_shift_cval = float("nan")

# Run field-star subtraction in parallel
star_subtraction_parallel = True
# Number of simultaneous subtraction tasks
star_subtraction_max_workers = 4

**Measurement settings**

In [ ]:
# Target position. Set both values to numbers to use a manual fixed position.
# Set both to None to detect the target automatically from star-subtracted data.
target_x = 3501.0
target_y = 2178.0

# Keep the same target position in every frame. This should normally be True
# for GEO satellite-tracking data.
fix_target_position = True

# Optional small centroid refinement. This is ignored when
# fix_target_position=True.
centroid_half_size = 8

# Aperture photometry settings [pixel]
aperture_radius    = 18.0
annulus_r_in       = 24.0
annulus_r_out      = 30.0

# Automatic target detection settings
detection_sigma            = 6.0
detection_cutout_half_size = 25

**Software metadata**<br>
Please DO NOT change from original.

In [ ]:
# Output software metadata
editor_app_name  = "SatPhotometry GEO SatTrack"
editor_version = "1.0.0"
creator_app_name = "SatPhotometry GEO SatTrack"
creator_version  = "1.0.0"

### Import and initial settings

In [ ]:
import json
import re
import shutil
import warnings
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

from astropy.io import fits
from astropy.stats import SigmaClip, sigma_clipped_stats
from astropy.table import Table
from astropy.time import Time, TimeDelta
from astropy.wcs import WCS
from astropy.wcs.wcs import FITSFixedWarning
from photutils.aperture import (
    ApertureStats,
    CircularAnnulus,
    CircularAperture,
    aperture_photometry,
)
from photutils.centroids import centroid_2dg
from tqdm.auto import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

from satphotometry import astrometry
from satphotometry import serparser

warnings.simplefilter("ignore", FITSFixedWarning)

### General utilities

In [ ]:
def get_fits_files(directory):
    """Return non-hidden FITS files directly contained in a directory."""
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(f"FITS directory does not exist: {directory}")

    files = [
        p for p in directory.iterdir()
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in {".fit", ".fits", ".fts"}
    ]
    files.sort(key=lambda p: p.name.casefold())
    if not files:
        raise FileNotFoundError(f"No FITS files were found in: {directory}")
    return files


def recreate_directory(directory):
    """Delete and recreate a processing directory."""
    directory = Path(directory)
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def get_observation_time(header):
    """Return exposure start, midpoint, and duration from a FITS header."""
    date_obs = header.get("DATE-OBS", header.get("DATEOBS"))
    if date_obs is None:
        raise KeyError("The FITS header contains neither DATE-OBS nor DATEOBS.")
    if "EXPTIME" not in header:
        raise KeyError("The FITS header does not contain EXPTIME.")

    exptime = float(header["EXPTIME"])
    if not np.isfinite(exptime) or exptime <= 0:
        raise ValueError(f"Invalid EXPTIME: {exptime}")

    try:
        time_start = Time(str(date_obs).strip(), format="fits", scale="utc")
    except ValueError:
        time_start = Time(str(date_obs).strip(), scale="utc")

    time_mid = time_start + TimeDelta(exptime / 2.0, format="sec")
    return time_start, time_mid, exptime


def windows_ticks_to_time(value):
    """Convert SER/Windows ticks since 0001-01-01 to Astropy Time."""
    unix_seconds = float(value) / 10_000_000.0 - 62_135_596_800.0
    return Time(unix_seconds, format="unix", scale="utc")


def parse_number(text):
    """Return the first floating-point number contained in text."""
    if text is None:
        return None
    match = re.search(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(text))
    return None if match is None else float(match.group(0))


def parse_sharpcap_settings(path):
    """Read commonly used fields from a SharpCap CameraSettings.txt file."""
    path = Path(path)
    if not path.exists():
        return {}

    settings = {}
    with path.open("r", encoding="utf-8", errors="replace") as file:
        for line in file:
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip('"')

    result = {}
    result["camera_name"] = settings.get("Camera") or settings.get("CameraName")
    result["camera_id"] = settings.get("CameraSerialNumber")

    binning = settings.get("Binning")
    if binning and "x" in binning.lower():
        x_bin, y_bin = re.split("x", binning, maxsplit=1, flags=re.IGNORECASE)
        result["x_binning"] = int(parse_number(x_bin))
        result["y_binning"] = int(parse_number(y_bin))

    exposure_text = settings.get("Exposure")
    exposure_value = parse_number(exposure_text)
    if exposure_value is not None:
        if "ms" in exposure_text.lower():
            exposure_value /= 1000.0
        elif "us" in exposure_text.lower() or "µs" in exposure_text.lower():
            exposure_value /= 1_000_000.0
        result["exposure"] = exposure_value

    key_map = {
        "Gain": "gain",
        "Offset": "offset",
        "Temperature": "ccd_temperature",
        "Target Temperature": "set_temperature",
    }
    for source_key, destination_key in key_map.items():
        value = parse_number(settings.get(source_key))
        if value is not None:
            result[destination_key] = value

    if "SharpCapVersion" in settings:
        result["swcreate"] = f"SharpCap {settings['SharpCapVersion']}"
    return result


### SER conversion and calibration

In [ ]:
def convert_ser_to_fits(
    ser_path,
    output_directory,
    *,
    start_frame=None,
    end_frame=None,
    exposure_seconds,
    gain=None,
    egain=None,
    egainsav=None,
    offset=None,
    x_binning=1,
    y_binning=1,
    camera_name=None,
    camera_id=None,
    ccd_temperature=None,
    set_temperature=None,
    observer=None,
    observatory=None,
    object_name=None,
    intl_des=None,
    norad_id=None,
):
    """Convert selected SER frames to FITS using satphotometry.serparser."""
    ser_path = Path(ser_path)
    output_directory = recreate_directory(output_directory)
    ser_file = serparser.Serfile(str(ser_path))
    frame_count = int(ser_file.getLength())

    first = 1 if start_frame is None else int(start_frame)
    last = frame_count if end_frame is None else int(end_frame)
    if first < 1 or last > frame_count or first > last:
        raise ValueError(
            f"Invalid SER frame range: {first}--{last}; total={frame_count}"
        )

    camera_settings_path = ser_path.with_suffix(".CameraSettings.txt")
    camera_settings = parse_sharpcap_settings(camera_settings_path)

    exposure_seconds = camera_settings.get("exposure", exposure_seconds)
    gain = camera_settings.get("gain", gain)
    offset = camera_settings.get("offset", offset)
    x_binning = camera_settings.get("x_binning", x_binning)
    y_binning = camera_settings.get("y_binning", y_binning)
    camera_name = camera_settings.get("camera_name", camera_name)
    camera_id = camera_settings.get("camera_id", camera_id)
    ccd_temperature = camera_settings.get("ccd_temperature", ccd_temperature)
    set_temperature = camera_settings.get("set_temperature", set_temperature)
    swcreate = camera_settings.get("swcreate", "satphotometry.serparser")

    try:
        timestamps = ser_file.readTrailFromHeader()
        if len(timestamps) != frame_count:
            timestamps = None
    except Exception:
        timestamps = None

    if timestamps is None:
        raise RuntimeError(
            "The SER file does not contain a usable per-frame timestamp trail. "
            "Reliable light-curve timing requires SER timestamps."
        )

    output_files = []
    stem = ser_path.stem
    for zero_index in tqdm(range(first - 1, last), desc="Converting SER to FITS"):
        output_path = output_directory / f"{stem}_{zero_index + 1:05d}.fits"
        ser_file.setCurrentPosition(zero_index)
        ser_file.saveFit(str(output_path))

        with fits.open(output_path, mode="update", memmap=False) as hdul:
            header = hdul[0].header
            time_start = windows_ticks_to_time(timestamps[zero_index])
            header["DATE-OBS"] = (time_start.fits, "Exposure start time UTC")
            header["EXPTIME"] = (float(exposure_seconds), "Exposure time [s]")
            header["XBINNING"] = int(x_binning)
            header["YBINNING"] = int(y_binning)
            header["SWCREATE"] = swcreate

            optional_header = {
                "GAIN": gain,
                "EGAIN": egain,
                "EGAINSAV": egainsav,
                "OFFSET": offset,
                "INSTRUME": camera_name,
                "CAMID": camera_id,
                "CCD-TEMP": ccd_temperature,
                "SET-TEMP": set_temperature,
                "OBSERVER": observer,
                "OBSERVAT": observatory,
                "OBJECT": object_name,
                "INTL-DES": intl_des,
                "NORAD-ID": norad_id,
            }
            for key, value in optional_header.items():
                if value is not None:
                    header[key] = value
            hdul.flush(output_verify="silentfix")

        output_files.append(output_path)
    return output_files


def create_master_dark(path):
    """Read a single dark frame or median-combine dark frames in a directory."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def create_master_flat(path, master_flat_dark=None):
    """Read and normalize a single flat or a median-combined flat stack."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        if master_flat_dark is not None:
            data = data - master_flat_dark
        stack.append(data)
    master = np.nanmedian(np.stack(stack), axis=0)
    norm = np.nanmedian(master[np.isfinite(master)])
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("The master flat cannot be normalized.")
    return master / norm


def calibrate_fits_files(fits_files, dark=None, flat=None):
    """Apply dark subtraction and flat-field correction in place."""
    if dark is None and flat is None:
        return
    for file_path in tqdm(fits_files, desc="Calibrating FITS frames"):
        with fits.open(file_path, mode="update", memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
            if dark is not None:
                data = data - dark
            if flat is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    data = data / flat
            hdul[0].data = data
            hdul.flush(output_verify="silentfix")

### Plate solving

In [ ]:
def build_astrometry_options(output_path, work_directory, previous_radec=None):
    """Build solve-field options accepted by satphotometry.astrometry.platesolve."""
    work_directory = Path(work_directory)
    work_directory.mkdir(parents=True, exist_ok=True)
    options = {
        "-N": str(output_path),
        "--dir": str(work_directory),
        "--downsample": int(astrometry_downsample),
    }
    options.update(astrometry_extra_options)

    if astrometry_scale_low_arcsec_per_pixel is not None:
        options["--scale-low"] = astrometry_scale_low_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"
    if astrometry_scale_high_arcsec_per_pixel is not None:
        options["--scale-high"] = astrometry_scale_high_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"

    if previous_radec is not None:
        options["--ra"] = float(previous_radec[0])
        options["--dec"] = float(previous_radec[1])
        options["--radius"] = float(astrometry_search_radius_deg)
    return options

def plate_solve_single_frame(task):
    """
    Plate-solve one frame in a separate process.
    """

    (
        frame_index,
        file_path,
        solved_path,
        frame_work,
        reference_radec,
    ) = task

    file_path = Path(file_path)
    solved_path = Path(solved_path)
    frame_work = Path(frame_work)

    frame_work.mkdir(
        parents=True,
        exist_ok=True,
    )

    options = build_astrometry_options(
        output_path=solved_path,
        work_directory=frame_work,
        previous_radec=reference_radec,
    )

    try:
        result_path, solved = astrometry.platesolve(
            str(file_path),
            options,
            async_process=False,
            jupyter_env=True,
        )

        if not solved:
            return {
                "frame_index": frame_index,
                "source_path": file_path,
                "solved_path": Path(result_path),
                "solved": False,
                "error": "solve-field did not solve the frame.",
            }

        result_path = Path(result_path)

        if not result_path.exists():
            return {
                "frame_index": frame_index,
                "source_path": file_path,
                "solved_path": result_path,
                "solved": False,
                "error": "The solved FITS file was not created.",
            }

        return {
            "frame_index": frame_index,
            "source_path": file_path,
            "solved_path": result_path,
            "solved": True,
            "error": None,
        }

    except Exception as error:
        return {
            "frame_index": frame_index,
            "source_path": file_path,
            "solved_path": solved_path,
            "solved": False,
            "error": str(error),
        }

def solve_reference_frame(
    file_path,
    solved_directory,
    work_directory,
):
    """
    Solve one reference frame without an RA/Dec constraint.
    """

    file_path = Path(file_path)

    solved_path = (
        Path(solved_directory)
        / file_path.name
    )

    frame_work = (
        Path(work_directory)
        / f"{file_path.stem}_reference"
    )

    options = build_astrometry_options(
        output_path=solved_path,
        work_directory=frame_work,
        previous_radec=None,
    )

    result_path, solved = astrometry.platesolve(
        str(file_path),
        options,
        async_process=True,
        jupyter_env=True,
    )

    if not solved or not Path(result_path).exists():
        raise RuntimeError(
            "The reference frame could not be plate-solved."
        )

    with fits.open(
        result_path,
        memmap=False,
    ) as hdul:
        data_shape = hdul[0].data.shape
        solved_header = hdul[0].header.copy()

    reference_wcs = WCS(
        solved_header
    )

    center_x = (
        data_shape[1] - 1
    ) / 2.0

    center_y = (
        data_shape[0] - 1
    ) / 2.0

    center_sky = reference_wcs.pixel_to_world(
        center_x,
        center_y,
    )

    reference_radec = (
        float(center_sky.ra.deg),
        float(center_sky.dec.deg),
    )

    return (
        Path(result_path),
        reference_radec,
    )

def plate_solve_files_parallel(
    fits_files,
    solved_directory,
    work_directory,
    max_workers=4,
):
    """
    Plate-solve frames in parallel using one common
    RA/Dec search center.
    """

    solved_directory = recreate_directory(
        solved_directory
    )

    work_directory = recreate_directory(
        work_directory
    )

    fits_files = [
        Path(file_path)
        for file_path in fits_files
    ]

    if len(fits_files) == 0:
        raise ValueError(
            "No FITS files were supplied."
        )

    # Use the middle frame as the reference because it is less
    # likely to lie at the edge of the total star-motion range.
    reference_index = (
        len(fits_files) // 2
    )

    reference_file = fits_files[
        reference_index
    ]

    (
        reference_solved_path,
        reference_radec,
    ) = solve_reference_frame(
        file_path=reference_file,
        solved_directory=solved_directory,
        work_directory=work_directory,
    )

    print(
        "Reference astrometry center: "
        f"RA={reference_radec[0]:.6f} deg, "
        f"Dec={reference_radec[1]:.6f} deg"
    )

    tasks = []

    for frame_index, file_path in enumerate(
        fits_files
    ):
        solved_path = (
            solved_directory
            / file_path.name
        )

        frame_work = (
            work_directory
            / file_path.stem
        )

        tasks.append(
            (
                frame_index,
                str(file_path),
                str(solved_path),
                str(frame_work),
                reference_radec,
            )
        )

    raw_results = []

    with ThreadPoolExecutor(
        max_workers=max_workers,
    ) as executor:
        future_to_index = {
            executor.submit(
                plate_solve_single_frame,
                task,
            ): task[0]
            for task in tasks
        }

        for future in tqdm(
            as_completed(future_to_index),
            total=len(future_to_index),
            desc="Plate solving",
        ):
            frame_index = future_to_index[
                future
            ]

            try:
                result = future.result()
            except Exception as error:
                result = {
                    "frame_index": frame_index,
                    "source_path": (
                        fits_files[frame_index]
                    ),
                    "solved_path": (
                        solved_directory
                        / fits_files[frame_index].name
                    ),
                    "solved": False,
                    "error": str(error),
                }

            raw_results.append(
                result
            )

    # Restore chronological frame order.
    raw_results.sort(
        key=lambda item: item[
            "frame_index"
        ]
    )

    records = []

    for result in raw_results:
        wcs = None
        header = None

        if (
            result["solved"]
            and result["solved_path"].exists()
        ):
            try:
                with fits.open(
                    result["solved_path"],
                    memmap=False,
                ) as hdul:
                    header = (
                        hdul[0].header.copy()
                    )

                wcs = WCS(
                    header
                )

            except Exception as error:
                result["solved"] = False
                result["error"] = (
                    "The solved FITS file could not be read: "
                    f"{error}"
                )

        if result["error"] is not None:
            print(
                "Plate solve failed for "
                f"{result['source_path'].name}: "
                f"{result['error']}"
            )

        records.append(
            {
                "source_path": (
                    result["source_path"]
                ),
                "solved_path": (
                    result["solved_path"]
                ),
                "solved": bool(
                    wcs is not None
                ),
                "wcs": wcs,
                "header": header,
            }
        )

    return records

def interpolate_star_positions(records):
    """Interpolate a common sky coordinate's pixel position in failed frames."""
    valid_indices = [i for i, record in enumerate(records) if record["wcs"] is not None]
    if len(valid_indices) < 2:
        raise RuntimeError("At least two successfully plate-solved frames are required.")

    reference_index = valid_indices[0]
    reference_record = records[reference_index]
    with fits.open(reference_record["source_path"], memmap=False) as hdul:
        ny, nx = np.asarray(hdul[0].data).shape
    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0
    common_sky = reference_record["wcs"].pixel_to_world(center_x, center_y)

    x_values = np.full(len(records), np.nan, dtype=float)
    y_values = np.full(len(records), np.nan, dtype=float)
    for i, record in enumerate(records):
        if record["wcs"] is not None:
            x_values[i], y_values[i] = record["wcs"].world_to_pixel(common_sky)

    indices = np.arange(len(records), dtype=float)
    x_values = np.interp(indices, valid_indices, x_values[valid_indices])
    y_values = np.interp(indices, valid_indices, y_values[valid_indices])
    return x_values, y_values

### Star subtraction

In [ ]:
def choose_reference_index(index, frame_count, interval):
    """Choose a separated frame while retaining every science frame."""
    if interval < 1:
        raise ValueError("star_reference_interval must be at least 1.")
    if frame_count <= interval:
        raise ValueError(
            "The number of frames must exceed star_reference_interval."
        )
    later = index + interval
    if later < frame_count:
        return later
    return index - interval


def shift_image(image, delta_x, delta_y):
    """Shift an image in detector x/y coordinates without integer wrapping."""
    finite = np.isfinite(image)
    filled = np.where(finite, image, 0.0)
    shifted_data = ndimage.shift(
        filled,
        shift=(delta_y, delta_x),
        order=star_shift_order,
        mode=star_shift_mode,
        cval=0.0,
        prefilter=star_shift_order > 1,
    )
    shifted_weight = ndimage.shift(
        finite.astype(float),
        shift=(delta_y, delta_x),
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )
    shifted_data[shifted_weight < 0.99] = star_shift_cval
    return shifted_data

def subtract_field_stars_single_frame(
    index,
    records,
    star_x,
    star_y,
    output_directory,
):
    """
    Subtract a star-aligned reference frame from one frame.

    Parameters
    ----------
    index : int
        Index of the current science frame.

    records : list[dict]
        Astrometry records for all frames.

    star_x, star_y : ndarray
        Interpolated detector coordinates of the common
        celestial position in every frame.

    output_directory : pathlib.Path
        Directory in which the star-subtracted FITS file
        will be saved.

    Returns
    -------
    output_record : dict
        Updated record containing the star-subtracted FITS
        path and reference-frame information.
    """

    frame_count = len(records)

    record = records[index]

    reference_index = choose_reference_index(
        index,
        frame_count,
        int(star_reference_interval),
    )

    reference_record = records[
        reference_index
    ]

    # Read the current science frame.
    with fits.open(
        record["source_path"],
        memmap=False,
    ) as hdul:
        data = np.asarray(
            hdul[0].data,
            dtype=float,
        )

        header = hdul[0].header.copy()

    # Read the reference frame.
    with fits.open(
        reference_record["source_path"],
        memmap=False,
    ) as hdul:
        reference_data = np.asarray(
            hdul[0].data,
            dtype=float,
        )

    # Shift the reference image so that its field stars
    # coincide with those in the current frame.
    delta_x = (
        star_x[index]
        - star_x[reference_index]
    )

    delta_y = (
        star_y[index]
        - star_y[reference_index]
    )

    aligned_reference = shift_image(
        reference_data,
        delta_x,
        delta_y,
    )

    # Subtract the aligned field-star image.
    star_subtracted = (
        data
        - aligned_reference
    )

    output_path = (
        Path(output_directory)
        / record["source_path"].name
    )

    header["STARSUB"] = (
        True,
        "Field-star reference subtraction applied",
    )

    header["REFINDEX"] = (
        int(reference_index),
        "Zero-based reference frame index",
    )

    header["REFDX"] = (
        float(delta_x),
        "Reference shift in detector x [pixel]",
    )

    header["REFDY"] = (
        float(delta_y),
        "Reference shift in detector y [pixel]",
    )

    fits.writeto(
        output_path,
        star_subtracted,
        header,
        overwrite=True,
        output_verify="silentfix",
    )

    return {
        **record,
        "photometry_path": output_path,
        "reference_index": int(
            reference_index
        ),
        "reference_shift_x": float(
            delta_x
        ),
        "reference_shift_y": float(
            delta_y
        ),
    }

def subtract_field_stars(
    records,
    output_directory,
    parallel=True,
    max_workers=4,
):
    """
    Subtract star-aligned reference frames from all
    satellite-tracking frames.

    Parameters
    ----------
    records : list[dict]
        Astrometry records for all frames.

    output_directory : str or pathlib.Path
        Output directory for star-subtracted FITS files.

    parallel : bool, optional
        If True, process frames concurrently.

    max_workers : int, optional
        Maximum number of simultaneous subtraction tasks.

    Returns
    -------
    output_records : list[dict]
        Records in the original chronological frame order.
    """

    output_directory = recreate_directory(
        output_directory
    )

    star_x, star_y = (
        interpolate_star_positions(
            records
        )
    )

    frame_count = len(records)

    if frame_count == 0:
        return []

    max_workers = int(
        max_workers
    )

    if max_workers < 1:
        raise ValueError(
            "star_subtraction_max_workers must be at least 1."
        )

    # --------------------------------------------------------
    # Sequential processing
    # --------------------------------------------------------

    if (
        not parallel
        or max_workers == 1
    ):
        output_records = []

        for index in tqdm(
            range(frame_count),
            desc="Subtracting field stars",
        ):
            output_record = (
                subtract_field_stars_single_frame(
                    index=index,
                    records=records,
                    star_x=star_x,
                    star_y=star_y,
                    output_directory=(
                        output_directory
                    ),
                )
            )

            output_records.append(
                output_record
            )

        return output_records

    # --------------------------------------------------------
    # Parallel processing
    # --------------------------------------------------------

    output_records = [
        None
    ] * frame_count

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        future_to_index = {
            executor.submit(
                subtract_field_stars_single_frame,
                index,
                records,
                star_x,
                star_y,
                output_directory,
            ): index
            for index in range(
                frame_count
            )
        }

        for future in tqdm(
            as_completed(
                future_to_index
            ),
            total=frame_count,
            desc="Subtracting field stars",
        ):
            index = future_to_index[
                future
            ]

            try:
                output_records[index] = (
                    future.result()
                )

            except Exception as error:
                raise RuntimeError(
                    "Field-star subtraction failed for "
                    f"frame {index}: "
                    f"{records[index]['source_path'].name}"
                ) from error

    return output_records

### Photometry

In [ ]:
def detect_brightest_positive_source(
    image,
    detection_sigma=6.0,
    cutout_half_size=25,
):
    """Detect the brightest positive point-like residual in a 2-D image."""
    image = np.asarray(image, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError("The detection image contains no finite pixels.")
    _, median, std = sigma_clipped_stats(image[finite], sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        raise ValueError("Could not estimate detection-image noise.")

    cleaned = np.where(finite, image, median)
    threshold = median + detection_sigma * std
    labels, number = ndimage.label(cleaned > threshold)
    if number == 0:
        raise RuntimeError("No target candidate was detected.")

    best_label = None
    best_flux = -np.inf
    for label_number in range(1, number + 1):
        region = labels == label_number
        flux = np.sum(np.clip(cleaned[region] - median, 0, None))
        if flux > best_flux:
            best_flux = flux
            best_label = label_number

    yy, xx = np.nonzero(labels == best_label)
    rough_x = float(np.mean(xx))
    rough_y = float(np.mean(yy))
    ny, nx = image.shape
    x0 = max(0, int(round(rough_x)) - cutout_half_size)
    x1 = min(nx, int(round(rough_x)) + cutout_half_size + 1)
    y0 = max(0, int(round(rough_y)) - cutout_half_size)
    y1 = min(ny, int(round(rough_y)) + cutout_half_size + 1)
    cutout = cleaned[y0:y1, x0:x1]
    _, local_median, _ = sigma_clipped_stats(cutout, sigma=3.0, maxiters=5)
    source = np.clip(cutout - local_median, 0, None)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def build_target_detection_image(records, sample_count=30):
    """Median-combine positive star-subtracted samples for robust target detection."""
    if not records:
        raise ValueError("No star-subtracted records are available.")
    sample_indices = np.unique(
        np.linspace(0, len(records) - 1, min(sample_count, len(records))).astype(int)
    )
    stack = []
    for index in sample_indices:
        with fits.open(records[index]["photometry_path"], memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def refine_centroid(image, x, y, half_size=8):
    """Refine a source centroid locally with a 2-D Gaussian centroid."""
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape
    xc = int(round(float(x)))
    yc = int(round(float(y)))
    x0 = max(0, xc - half_size)
    x1 = min(nx, xc + half_size + 1)
    y0 = max(0, yc - half_size)
    y1 = min(ny, yc + half_size + 1)
    cutout = image[y0:y1, x0:x1]
    finite = np.isfinite(cutout)
    if np.count_nonzero(finite) < 9:
        raise ValueError("Too few finite pixels for centroid refinement.")
    _, median, _ = sigma_clipped_stats(cutout[finite], sigma=3.0, maxiters=5)
    source = np.where(finite, np.clip(cutout - median, 0, None), 0.0)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def measure_object(image, x_object, y_object, aperture_radius, annulus_r_in, annulus_r_out):
    """Perform circular-aperture photometry with a sigma-clipped annulus."""
    image = np.asarray(image, dtype=float)
    position = (float(x_object), float(y_object))
    aperture = CircularAperture(position, r=float(aperture_radius))
    annulus = CircularAnnulus(
        position,
        r_in=float(annulus_r_in),
        r_out=float(annulus_r_out),
    )
    phot_table = aperture_photometry(image, aperture, method="exact")
    aperture_sum = float(phot_table["aperture_sum"][0])
    annulus_stats = ApertureStats(
        image,
        annulus,
        sigma_clip=SigmaClip(sigma=3.0, maxiters=5),
    )
    background_median = float(annulus_stats.median)
    aperture_area = float(aperture.area)
    background_sum = background_median * aperture_area
    object_flux = aperture_sum - background_sum
    return {
        "aperture_sum": aperture_sum,
        "background_median": background_median,
        "background_sum": background_sum,
        "aperture_area": aperture_area,
        "object_flux": object_flux,
    }


def read_photometric_zeropoint(magzero_path):
    """
    Prefer photometric_zeropoint_elec. If unavailable, use
    photometric_zeropoint_exp without detector-gain correction.
    """
    with Path(magzero_path).open("r", encoding="utf-8") as file:
        data = json.load(file)
    source = data.get("phot_params", data)

    zp_elec = source.get("photometric_zeropoint_elec", source.get("ZP_elec"))
    try:
        zp_elec = float(zp_elec)
    except (TypeError, ValueError):
        zp_elec = np.nan
    if np.isfinite(zp_elec):
        return data, zp_elec, "elec"

    zp_exp = source.get("photometric_zeropoint_exp", source.get("ZP_exp"))
    try:
        zp_exp = float(zp_exp)
    except (TypeError, ValueError):
        zp_exp = np.nan
    if np.isfinite(zp_exp):
        return data, zp_exp, "exp"

    raise KeyError(
        "Neither a finite photometric_zeropoint_elec nor a finite "
        "photometric_zeropoint_exp is available."
    )


def calculate_apparent_magnitude(object_flux, exptime, header, zeropoint, mode):
    """Calculate apparent magnitude with the selected zeropoint convention."""
    if not np.isfinite(object_flux) or object_flux <= 0:
        return np.nan, np.nan, np.nan

    instrumental_mag = -2.5 * np.log10(object_flux)
    if mode == "elec":
        if "EGAINSAV" not in header:
            raise KeyError(
                "photometric_zeropoint_elec is being used, but EGAINSAV is "
                "missing from the science FITS header."
            )
        egainsav = float(header["EGAINSAV"])
        if not np.isfinite(egainsav) or egainsav <= 0:
            raise ValueError(f"Invalid EGAINSAV: {egainsav}")
        apparent_mag = (
            instrumental_mag
            + 2.5 * np.log10(exptime)
            - 2.5 * np.log10(egainsav)
            + zeropoint
        )
    elif mode == "exp":
        egainsav = np.nan
        apparent_mag = instrumental_mag + 2.5 * np.log10(exptime) + zeropoint
    else:
        raise ValueError(f"Unknown zeropoint mode: {mode}")
    return instrumental_mag, apparent_mag, egainsav

### Output

In [ ]:
def json_value(value):
    """Convert NumPy/Astropy scalar values to JSON-compatible values."""
    if np.ma.is_masked(value):
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def export_kupt_lightcurve_json(
    table,
    output_path,
    fits_header,
    magzero_data,
    zeropoint_mode,
):
    """Write a compact KUPT-lightcurveJSON version 2 product."""
    valid_rows = [row for row in table if str(row["time_start_utc"]).strip()]
    if not valid_rows:
        raise ValueError("No valid timestamp is available for JSON output.")

    first = valid_rows[0]
    last = valid_rows[-1]
    start = Time(str(first["time_start_utc"]), scale="utc")
    end = Time(str(last["time_start_utc"]), scale="utc") + TimeDelta(
        float(last["exptime_seconds"]), format="sec"
    )
    phot_source = magzero_data.get("phot_params", magzero_data)

    data_rows = []
    for row in table:
        data_rows.append({
            "id": int(row["frame_index"]),
            "time_start_utc": str(row["time_start_utc"]) or None,
            "time_mid_utc": str(row["time_mid_utc"]) or None,
            "elapsed_seconds": json_value(row["elapsed_seconds"]),
            "x_centroid": json_value(row["x_centroid"]),
            "y_centroid": json_value(row["y_centroid"]),
            "aperture_sum": json_value(row["aperture_sum_adu"]),
            "bkg_med_pix": json_value(row["background_median_adu_per_pixel"]),
            "object_flux": json_value(row["object_flux_adu"]),
            "mag_in": json_value(row["instrumental_mag"]),
            "mag_app": json_value(row["apparent_mag"]),
            "status": str(row["status"]),
            "comment": {
                "general": [
                    f"Source FITS file: {row['filename']}",
                    f"Star reference frame index: {row['star_reference_index']}",
                    f"Zeropoint mode: {zeropoint_mode}",
                ]
            },
        })

    output = {
        "header": {
            "object": {
                "objectName": object_name,
                "intlDES": intl_des,
                "noradID": None if norad_id is None else str(norad_id),
            },
            "startDateTime": start.isot,
            "endDateTime": end.isot,
            "length": len(table),
            "captureSettings": {
                "Xnaxis": json_value(fits_header.get("NAXIS1")),
                "Ynaxis": json_value(fits_header.get("NAXIS2")),
                "Xbinning": json_value(fits_header.get("XBINNING", 1)),
                "Ybinning": json_value(fits_header.get("YBINNING", 1)),
                "exposure": json_value(fits_header.get("EXPTIME")),
                "gain": json_value(fits_header.get("GAIN")),
                "egain": json_value(fits_header.get("EGAIN")),
                "egainsav": json_value(fits_header.get("EGAINSAV")),
                "offset": json_value(fits_header.get("OFFSET")),
                "camName": json_value(fits_header.get("INSTRUME")),
                "camID": json_value(fits_header.get("CAMID")),
                "swcreate": json_value(fits_header.get("SWCREATE")),
            },
            "photParams": {
                "pixel_area_arcsec2": json_value(phot_source.get("pixel_area_arcsec2")),
                "aperture_area_arcsec2": json_value(phot_source.get("aperture_area_arcsec2")),
                "ZP": json_value(phot_source.get("photometric_zeropoint")),
                "ZP_exp": json_value(phot_source.get("photometric_zeropoint_exp")),
                "ZP_elec": json_value(phot_source.get("photometric_zeropoint_elec")),
                "ZP_scatter": json_value(phot_source.get("zeropoint_scatter")),
                "selected_mode": zeropoint_mode,
            },
            "observer": {
                "observerName": observer_name,
                "observatory": observatory,
            },
            "editor": {
                "appName": editor_app_name,
                "version": editor_version,
            },
            "measureSettings": {
                "measureFieldShape": "CIRCLE",
                "measureFieldSize": {
                    "aperture_radius_pixel": float(aperture_radius),
                    "annulus_r_in_pixel": float(annulus_r_in),
                    "annulus_r_out_pixel": float(annulus_r_out),
                },
            },
        },
        "data": data_rows,
        "createdBy": {
            "appName": creator_app_name,
            "version": creator_version,
            "built": int(datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")),
        },
        "dataType": "KUPT-lightcurveJSON",
        "version": 2,
        "comment": {
            "general": [
                "GEO satellite-tracking photometry with WCS-based field-star subtraction."
            ],
            "TLE": [],
        },
    }

    # Remove absent optional capture settings while retaining mandatory dimensions.
    output["header"]["captureSettings"] = {
        key: value
        for key, value in output["header"]["captureSettings"].items()
        if value is not None
    }
    output["header"]["photParams"] = {
        key: value
        for key, value in output["header"]["photParams"].items()
        if value is not None
    }

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(output, file, ensure_ascii=False, indent=2, allow_nan=False)
    return output

### Main pipeline

In [ ]:
def create_geo_lightcurve():
    """Run the complete GEO satellite-tracking photometry pipeline."""
    PATH_output.mkdir(parents=True, exist_ok=True)
    raw_fits_directory = PATH_output / "fits_raw"
    solved_directory = PATH_output / "fits_solved"
    astrometry_work_directory = PATH_output / "astrometry_work"
    star_subtracted_directory = PATH_output / "fits_star_subtracted"
    product_directory = PATH_output / "lightcurve"
    product_directory.mkdir(parents=True, exist_ok=True)

    if input_format.upper() == "SER":
        fits_files = convert_ser_to_fits(
            PATH_input,
            raw_fits_directory,
            start_frame=ser_start_frame,
            end_frame=ser_end_frame,
            exposure_seconds=ser_exposure_seconds,
            gain=ser_gain,
            egain=ser_egain,
            egainsav=ser_egainsav,
            offset=ser_offset,
            x_binning=ser_x_binning,
            y_binning=ser_y_binning,
            camera_name=ser_camera_name,
            camera_id=ser_camera_id,
            ccd_temperature=ser_ccd_temperature,
            set_temperature=ser_set_temperature,
            observer=observer_name,
            observatory=observatory,
            object_name=object_name,
            intl_des=intl_des,
            norad_id=norad_id,
        )
    elif input_format.upper() == "FITS":
        source_files = get_fits_files(PATH_input)
        recreate_directory(raw_fits_directory)
        fits_files = []
        for source in source_files:
            destination = raw_fits_directory / source.name
            shutil.copy2(source, destination)
            fits_files.append(destination)
    else:
        raise ValueError("input_format must be either 'SER' or 'FITS'.")

    master_flat_dark = create_master_dark(PATH_flat_dark)
    master_dark = create_master_dark(PATH_dark)
    master_flat = create_master_flat(PATH_flat, master_flat_dark)
    calibrate_fits_files(fits_files, dark=master_dark, flat=master_flat)

    if not run_astrometry and subtract_stars:
        raise ValueError("WCS plate solving is required for star subtraction.")

    if run_astrometry:
        if astrometry_parallel:
            astrometry_records = plate_solve_files_parallel(
                fits_files=fits_files,
                solved_directory=solved_directory,
                work_directory=astrometry_work_directory,
                max_workers=astrometry_max_workers,
            )
        else:
            astrometry_records = plate_solve_files(
                fits_files,
                solved_directory,
                astrometry_work_directory,
            )
    else:
        astrometry_records = [
            {
                "source_path": path,
                "solved_path": path,
                "solved": False,
                "wcs": None,
                "header": None,
            }
            for path in fits_files
        ]

    if subtract_stars:
        photometry_records = subtract_field_stars(
            records=astrometry_records,
            output_directory=(
                star_subtracted_directory
            ),
            parallel=(
                star_subtraction_parallel
            ),
            max_workers=(
                star_subtraction_max_workers
            ),
        )
    else:
        photometry_records = [
            {
                **record,
                "photometry_path": record["source_path"],
                "reference_index": -1,
                "reference_shift_x": np.nan,
                "reference_shift_y": np.nan,
            }
            for record in astrometry_records
        ]

    if target_x is None and target_y is None:
        detection_image = build_target_detection_image(photometry_records)
        x_fixed, y_fixed = detect_brightest_positive_source(
            detection_image,
            detection_sigma=detection_sigma,
            cutout_half_size=detection_cutout_half_size,
        )
        fits.writeto(
            PATH_output / "target_detection_image.fits",
            detection_image,
            overwrite=True,
        )
        print(f"Automatically detected target: x={x_fixed:.3f}, y={y_fixed:.3f}")
    elif target_x is not None and target_y is not None:
        x_fixed = float(target_x)
        y_fixed = float(target_y)
    else:
        raise ValueError("target_x and target_y must both be set or both be None.")

    magzero_data, zeropoint, zeropoint_mode = read_photometric_zeropoint(PATH_magzero)
    print(f"Using zeropoint mode: {zeropoint_mode}")

    results = []
    first_time_mid = None
    first_header = None
    previous_x = x_fixed
    previous_y = y_fixed

    for frame_index, record in enumerate(tqdm(photometry_records, desc="Measuring GEO target")):
        path = record["photometry_path"]
        status = "Success"
        try:
            with fits.open(path, memmap=False) as hdul:
                image = np.asarray(hdul[0].data, dtype=float)
                header = hdul[0].header.copy()
            if first_header is None:
                first_header = header.copy()

            time_start, time_mid, exptime = get_observation_time(header)
            if first_time_mid is None:
                first_time_mid = time_mid
            elapsed = float((time_mid - first_time_mid).to_value("sec"))

            if fix_target_position:
                x_object, y_object = x_fixed, y_fixed
                position_method = "fixed_position"
            else:
                x_object, y_object = refine_centroid(
                    image,
                    previous_x,
                    previous_y,
                    half_size=centroid_half_size,
                )
                previous_x, previous_y = x_object, y_object
                position_method = "local_centroid"

            phot = measure_object(
                image,
                x_object,
                y_object,
                aperture_radius,
                annulus_r_in,
                annulus_r_out,
            )
            instrumental_mag, apparent_mag, egainsav = calculate_apparent_magnitude(
                phot["object_flux"],
                exptime,
                header,
                zeropoint,
                zeropoint_mode,
            )
            if not np.isfinite(apparent_mag):
                status = "Non-positive flux"

            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": time_start.isot,
                "time_mid_utc": time_mid.isot,
                "elapsed_seconds": elapsed,
                "exptime_seconds": exptime,
                "egainsav_electron_per_adu": egainsav,
                "x_centroid": x_object,
                "y_centroid": y_object,
                "position_method": position_method,
                "star_reference_index": record["reference_index"],
                "star_reference_shift_x": record["reference_shift_x"],
                "star_reference_shift_y": record["reference_shift_y"],
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": phot["aperture_area"],
                "aperture_sum_adu": phot["aperture_sum"],
                "background_median_adu_per_pixel": phot["background_median"],
                "background_sum_adu": phot["background_sum"],
                "object_flux_adu": phot["object_flux"],
                "instrumental_mag": instrumental_mag,
                "apparent_mag": apparent_mag,
                "zeropoint_mode": zeropoint_mode,
                "status": status,
            })
        except Exception as error:
            print(f"Frame {frame_index} ({path.name}) failed: {error}")
            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": "",
                "time_mid_utc": "",
                "elapsed_seconds": np.nan,
                "exptime_seconds": np.nan,
                "egainsav_electron_per_adu": np.nan,
                "x_centroid": np.nan,
                "y_centroid": np.nan,
                "position_method": "",
                "star_reference_index": record["reference_index"],
                "star_reference_shift_x": record["reference_shift_x"],
                "star_reference_shift_y": record["reference_shift_y"],
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": np.nan,
                "aperture_sum_adu": np.nan,
                "background_median_adu_per_pixel": np.nan,
                "background_sum_adu": np.nan,
                "object_flux_adu": np.nan,
                "instrumental_mag": np.nan,
                "apparent_mag": np.nan,
                "zeropoint_mode": zeropoint_mode,
                "status": f"Error: {error}",
            })

    table = Table(rows=results)
    valid = [row for row in results if str(row["time_start_utc"]).strip()]
    if not valid:
        raise RuntimeError("No frame was measured successfully.")
    start = Time(valid[0]["time_start_utc"], scale="utc")
    end = Time(valid[-1]["time_start_utc"], scale="utc") + TimeDelta(
        valid[-1]["exptime_seconds"], format="sec"
    )
    base_name = (
        f"lightcurve_{object_name}_{intl_des}_"
        f"{start.datetime.strftime('%Y-%m-%d_%H%M%S')}_"
        f"{end.datetime.strftime('%H%M%S')}"
    ).replace(" ", "_")
    csv_path = product_directory / f"{base_name}.csv"
    json_path = product_directory / f"{base_name}.json"
    table.write(csv_path, format="ascii.csv", overwrite=True)
    json_output = export_kupt_lightcurve_json(
        table,
        json_path,
        first_header,
        magzero_data,
        zeropoint_mode,
    )
    print(f"Saved CSV:  {csv_path}")
    print(f"Saved JSON: {json_path}")
    return table, csv_path, json_path, json_output


### Run

In [ ]:
lightcurve_table, output_csv_path, output_json_path, kupt_json_data = (
    create_geo_lightcurve()
)

### Result

In [ ]:
elapsed_seconds = np.asarray(lightcurve_table["elapsed_seconds"], dtype=float)
apparent_mag = np.asarray(lightcurve_table["apparent_mag"], dtype=float)
valid = np.isfinite(elapsed_seconds) & np.isfinite(apparent_mag)

plt.figure(figsize=(9, 5))
plt.plot(
    elapsed_seconds[valid],
    apparent_mag[valid],
    marker="o",
    markersize=4,
    linewidth=1,
)
plt.xlabel("Elapsed time from first frame [s]")
plt.ylabel("Apparent magnitude [mag]")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()